# CAB320 Assignment 2 - Transfer Learning
Anthony Vanderkop, Thierry Peynot, Frederic Maire (Jupyter Notebook template: 2025)


## Instructions:
The functions and classes defined in this module will be called by the marker without modification. 
You should complete the functions and classes according to their specified interfaces.

No partial marks will be awarded for functions that do not meet the specifications of the interfaces.


In [41]:
### LIBRARY IMPORTS ###
import os
import numpy as np
import tensorflow as tf
import keras.applications as ka
import keras

## Task 1
Implement the my_team()function 

In [42]:
def my_team():
    '''
    Return the list of the team members of this assignment submission as a list
    of triplet of the form (student_number, first_name, last_name)
    
    '''
    return [
        (11301406, 'Joshua', 'Rapsey'),
        (11290803, 'Joshua', 'Hentscher'),
        (11631996, 'Cooper', 'Smith')
    ]

In [43]:
my_team()

[(11301406, 'Joshua', 'Rapsey'),
 (11290803, 'Joshua', 'Hentscher'),
 (11631996, 'Cooper', 'Smith')]

## Task 2
Download the small_flower_dataset from Canvas and load the data

In [44]:
def load_data(path='small_flower_dataset'):
    '''
    Load in the dataset from its home path. Path should be a string of the path
    to the home directory the dataset is found in. Returns the images as a
    numpy array and the associated class labels as a numpy array.
    '''
    class_names = [name for name in sorted(os.listdir(path)) if os.path.isdir(os.path.join(path, name))]
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp')
    images = []
    labels = []

    for class_index, class_name in enumerate(class_names):
        class_path = os.path.join(path, class_name)
        for file_name in sorted(os.listdir(class_path)):
            file_path = os.path.join(class_path, file_name)
            if os.path.isfile(file_path) and file_name.lower().endswith(image_extensions):
                image_bytes = tf.io.read_file(file_path)
                image = tf.image.decode_image(image_bytes, channels=3, expand_animations=False)
                image = tf.image.resize(image, (224, 224))
                images.append(image.numpy())
                labels.append(class_index)

    images = ka.mobilenet_v2.preprocess_input(np.asarray(images, dtype=np.float32))
    labels = np.asarray(labels, dtype=np.int64)
    return images, labels

In [45]:
dataset = load_data()

## Task 3
Prepare your training, validation and test sets for the non-accelerated version of transfer learning.

In [50]:
def split_data(X, Y, train_fraction, randomize=False, eval_set=True):
    """
    Split the data into training and testing sets. If eval_set is True, also create
    an evaluation dataset. There should be two outputs if eval_set there should
    be three outputs (train, test, eval), otherwise two outputs (train, test).
    
    To see what type train, test, and eval should be, refer to the inputs of 
    transfer_learning().
    
    The split is performed in a stratified way so each class keeps a similar
    proportion in the train, evaluation, and test sets.
    """
    X = np.asarray(X)
    Y = np.asarray(Y)
    rng = np.random.default_rng()

    train_indices = []
    eval_indices = []
    test_indices = []

    for class_value in np.unique(Y):
        class_indices = np.where(Y == class_value)[0]
        if randomize:
            class_indices = class_indices.copy()
            rng.shuffle(class_indices)

        class_size = len(class_indices)
        remaining_required = 2 if eval_set and class_size >= 3 else 1 if (not eval_set and class_size >= 2) else 0
        train_count = int(round(train_fraction * class_size))
        train_count = max(0, min(train_count, class_size - remaining_required))

        remaining_indices = class_indices[train_count:]
        if eval_set:
            eval_count = len(remaining_indices) // 2
            test_count = len(remaining_indices) - eval_count
            train_indices.extend(class_indices[:train_count])
            eval_indices.extend(remaining_indices[:eval_count])
            test_indices.extend(remaining_indices[eval_count:eval_count + test_count])
        else:
            train_indices.extend(class_indices[:train_count])
            test_indices.extend(remaining_indices)

    if randomize:
        rng.shuffle(train_indices)
        rng.shuffle(eval_indices)
        rng.shuffle(test_indices)

    train_set = (X[train_indices], Y[train_indices])
    test_set = (X[test_indices], Y[test_indices])

    if eval_set:
        eval_set = (X[eval_indices], Y[eval_indices])
        return train_set, eval_set, test_set

    return train_set, test_set

In [52]:
train_set, eval_set, test_set = split_data(dataset[0], dataset[1], 0.6, randomize=True, eval_set=True)

Report: Include details of how you have split the data to perform this training. Ensure the split is reasonable and does not introduce class imbalance during training

The data was split using a stratified 60/20/20 train/evaluation/test split.
Images were shuffled within each class before splitting so the five flower
classes stay balanced across all partitions.

## Task 4
Using the tf.keras.applications module download a pretrained MobileNetV2 network. 

In [53]:
def load_model():
    '''
    Load in a model using the tf.keras.applications model and return it.
    The returned model is a pretrained MobileNetV2 network with the ImageNet
    weights loaded.
    '''
    return ka.MobileNetV2(weights='imagenet', include_top=True, input_shape=(224, 224, 3))
    

In [54]:
model = load_model()

## Task 5
Replace the last layer of the downloaded neural network with a Dense layer of the appropriate shape for the 5 classes of the small flower dataset {(x1,t1), (x2,t2),..., (xm,tm)}.

## Task 6
Compile and train your model with an SGD optimizer using the following parameters learning_rate=0.01, momentum=0.0, nesterov=False. (NB: The SGD class description can be found at https://keras.io/api/optimizers/sgd/  )

In [57]:
def transfer_learning(train_set, eval_set, model, parameters):
    '''
    Implement and perform standard transfer learning here.

    Inputs:
        - train_set: list or tuple of the training images and labels in the
            form (images, labels) for training the classifier
        - eval_set: list or tuple of the images and labels used in evaluating
            the model during training, in the form (images, labels)
        - model: an instance of tf.keras.applications.MobileNetV2
        - parameters: list or tuple of parameters to use during training:
            (learning_rate, momentum, nesterov)


    Outputs:
        - model : an instance of tf.keras.applications.MobileNetV2

    '''
    learning_rate, momentum, nesterov = parameters
    train_images, train_labels = train_set
    eval_images, eval_labels = eval_set

    for layer in model.layers:
        layer.trainable = False

    classifier = keras.layers.Dense(5, activation='softmax', name='flower_predictions')(model.layers[-2].output)
    model = keras.Model(inputs=model.input, outputs=classifier)
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=learning_rate,
            momentum=momentum,
            nesterov=nesterov,
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    model.fit(
        train_images,
        train_labels,
        validation_data=(eval_images, eval_labels),
        epochs=10,
        batch_size=32,
        verbose=1,
    )
    return model

In [58]:
model = transfer_learning(train_set, eval_set, load_model(), (0.01, 0.0, False))

Epoch 1/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 6s 230ms/step - accuracy: 0.4383 - loss: 1.3650 - val_accuracy: 0.6600 - val_loss: 1.0306
Epoch 2/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 194ms/step - accuracy: 0.7350 - loss: 0.8435 - val_accuracy: 0.7500 - val_loss: 0.7799
Epoch 3/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 200ms/step - accuracy: 0.8017 - loss: 0.6540 - val_accuracy: 0.7700 - val_loss: 0.6815
Epoch 4/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 194ms/step - accuracy: 0.8167 - loss: 0.5555 - val_accuracy: 0.7850 - val_loss: 0.6309
Epoch 5/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 202ms/step - accuracy: 0.8517 - loss: 0.4881 - val_accuracy: 0.7800 - val_loss: 0.5882
Epoch 6/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 194ms/step - accuracy: 0.8617 - loss: 0.4412 - val_accuracy: 0.7950 - val_loss: 0.5625
Epoch 7/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 211ms/step - accuracy: 0.8850 - loss: 0.4020 - val_accuracy: 0.8050 - val_loss: 0.5425
Epoch 8/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 198ms/step - accuracy: 0.8950 - loss: 0.3691 - val_accuracy: 0.

## Task 7
Plot the training and validation errors and accuracies of standard transfer 

In [ ]:
## Your Code

## Task 8
Experiment with 3 different orders of magnitude for the learning rate. Plot the results and discuss in the below markdown cell

In [ ]:
## Your code

### Task 8 Analysis and discussion


## Task 9
Run the resulting classifier on your test dataset using results from the best learning rate you experimented with. Compute and display the confusion matrix. 

In [ ]:
## Your code

## Task 10
Compute the precision, recall, and f1 scores of your classifier on the test dataset using the best learning rate. Report on the results and comment. 

In [ ]:
## Your code

## Task 11
Perform k-fold validation on the dataset with k = 3. 

In [ ]:
def k_fold_validation(features, ground_truth, classifier, k=2):
    '''
    Inputs:
        - features: np.ndarray of features in the dataset
        - ground_truth: np.ndarray of class values associated with the features
        - fit_func: f
        - classifier: class object with both fit() and predict() methods which
        can be applied to subsets of the features and ground_truth inputs.
        - predict_func: function, calling predict_func(features) should return
        a numpy array of class predictions which can in turn be input to the 
        functions in this script to calculate performance metrics.
        - k: int, number of sub-sets to partition the data into. default is k=2
    Outputs:
        - avg_metrics: np.ndarray of shape (3, c) where c is the number of classes.
        The first row is the average precision for each class over the k
        validation steps. Second row is recall and third row is f1 score.
        - sigma_metrics: np.ndarray, each value is the standard deviation of 
        the performance metrics [precision, recall, f1_score]
    '''
    
    #split data
    ### YOUR CODE HERE ###
    
    #go through each partition and use it as a test set.
    for partition_no in range(k):
        #determine test and train sets
        ### YOUR CODE HERE###
        
        #fit model to training data and perform predictions on the test set
        classifier.fit(train_features, train_classes)
        predictions = classifier.predict(test_features)
        
        #calculate performance metrics
        ### YOUR CODE HERE###
    
    #perform statistical analyses on metrics
    ### YOUR CODE HERE###
    
    raise NotImplementedError
    return avg_metrics, sigma_metrics

In [ ]:
## Your code
# xx = k_fold_validation(xx, xx, xx, xx)

Comment on the results and any differences with the previous test-train split. 
Repeat with two different values for k and comment on the results. 

### Comments and analysis

## Task 12
With the best learning rate that you found in the previous task, add a non-zero momentum to the training with the SGD optimizer (consider 3 values for the momentum). Report on how your results change.  

In [ ]:
## Code

### Report

## Task 13
Now using “accelerated transfer learning”, repeat the training process (k-fold validation is optional this time). You should prepare your training, validation and test sets based on {(F(x1).t1), (F(x2),t2),...,(F(xm),tm)}, and re-do Task 12. 


In [ ]:
def accelerated_learning(train_set, eval_set, model, parameters):
    '''
    Implement and perform accelerated transfer learning here.

    Inputs:
        - train_set: list or tuple of the training images and labels in the
            form (images, labels) for training the classifier
        - eval_set: list or tuple of the images and labels used in evaluating
            the model during training, in the form (images, labels)
        - model: an instance of tf.keras.applications.MobileNetV2
        - parameters: list or tuple of parameters to use during training:
            (learning_rate, momentum, nesterov)


    Outputs:
        - model : an instance of tf.keras.applications.MobileNetV2

    '''
    raise NotImplementedError
    return model


Plot and comment on the results and differences against the standard implementation of transfer learning. 

In [ ]:
## Code

### Your Comments:

## Task 14
Use the results of all experiments to make suggestions for future work and recommendations for parameter values to anyone else who may be interested in a similar implementation of transfer learning. 

### Your answer: